In [7]:
#Step 1.1

import sklearn
from sklearn.model_selection import train_test_split
import pandas as pd
df = pd.read_csv('pokemon_numeric.csv')
df = df.drop(columns=['Name'])


In [2]:
#Step 1.2

x = df.drop(columns=['hasMegaEvolution'])
y = df['hasMegaEvolution'].astype(int)

In [3]:
#Step 2.2

print(y.value_counts())
print(y.value_counts(normalize=True))

hasMegaEvolution
0    675
1     46
Name: count, dtype: int64
hasMegaEvolution
0    0.9362
1    0.0638
Name: proportion, dtype: float64


In [ ]:
#Question 1:

#The distribution of class labels in the y array is 46 observations, or 6.38%
#The classes are very unbalanced since almost 94% of the samples belong to the x array while the
#rest belong to the y. It is more difficylt to build good predictive models this way because there is a large disproportion
# of 0 to 1s in this case and a model predicting 0 for every instance would be accurate 93.62% of the time.

In [8]:
#Step 2.2

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3,random_state=0)

In [14]:
#Step 2.3

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
#Question 2:

#We are only fitting the standardization object on the x_train because
#we wouldn't want the mdoel to know "understand the answers" by giving it
#the data on the x_test as well.

In [15]:
#Step 2.4

from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced',y=y_train)

In [ ]:
#Question 3:

#Conceptually, this would assign a multiplier to each observation based on its class
#It is necessary for MLPClassifier because it would optimize neural weights using sgd.

In [25]:
#Step 2.5

from sklearn.neural_network import MLPClassifier
modelNN = MLPClassifier(random_state=0,activation='logistic',solver='sgd',hidden_layer_sizes=8)

In [24]:
#Step 2.7

from sklearn.metrics import confusion_matrix
modelNN = MLPClassifier(random_state=0,activation='logistic',solver='sgd',hidden_layer_sizes=8,max_iter=2000,)
modelNN.fit(x_train,y_train,sample_weight=sample_weights)
predicted_test = modelNN.predict(x_test)
print(confusion_matrix(y_test,predicted_test,labels=[1,0]))

[[  8   6]
 [ 69 134]]


In [26]:
#Step 2.6

modelNN.fit(x_train,y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,hidden_layer_sizes,8
,activation,'logistic'
,solver,'sgd'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,0


In [ ]:
#Question 5:

#The problem with this confusion matrix is that there is more false positive
#than what there are actually supposed to be. Since the model detects over half of
# the mega evolutions with 8 / (8+6) = over 50%, there are a large proportion of false positives.
#The sample weights penalized missed positec cases heavily

In [29]:
#Step 3.3

import numpy as np
from sklearn.model_selection import GridSearchCV

modelNN = MLPClassifier(random_state=0,max_iter=2000,activation='logistic',solver='sgd')
params = {'hidden_layer_sizes': list(range(8, 21)),'learning_rate_init': [round(x, 1) for x in np.arange(0.1, 1.1, 0.1)]}
gs = GridSearchCV(estimator=modelNN,param_grid=params,cv=3,verbose=2,scoring='recall')
gs.fit(x_train, y_train, sample_weight=sample_weights)

#Question 6:
print("Best Parameters:", gs.best_params_)

Fitting 3 folds for each of 130 candidates, totalling 390 fits
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.4; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.4; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning

In [30]:
#Step 3.4
best_hidden_size = gs.best_params_['hidden_layer_sizes']
best_learning_rate = gs.best_params_['learning_rate_init']

modelNN_tuned = MLPClassifier(random_state=0,max_iter=2000,activation='logistic',solver='sgd',hidden_layer_sizes=best_hidden_size,learning_rate_init=best_learning_rate)

modelNN_tuned.fit(x_train, y_train, sample_weight=sample_weights)

predicted_test_tuned = modelNN_tuned.predict(x_test)

cm_tuned = confusion_matrix(y_test, predicted_test_tuned, labels=[1, 0])
print(cm_tuned)

[[  9   5]
 [ 62 141]]


In [ ]:
#Question 7:

#The higher learning rate sllows sgd to take larger steps when penalizing the minority class sample weights
# It would predict positive rates more carefully and increased the total count of true positives.

In [ ]:
#Question 8: 

# When recalls are optimized, it penalized false negatives without penalizing false positives
# and so to avoid missing any true 1s, the network lowers its effective threshold. This tradeoff means that both true
# positivies and false positives rise and cause precision to decline so a lot of non-mega evolutions are classified as mega evolutions.

In [33]:
#Step 4.2

from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score

base_nn = MLPClassifier(random_state=0,max_iter=2000,activation="logistic",solver="sgd",
    hidden_layer_sizes=gs.best_params_["hidden_layer_sizes"],
    learning_rate_init=gs.best_params_["learning_rate_init"],)

bagging_model = BaggingClassifier(estimator=base_nn, random_state=22)


bagging_model.fit(x_train, y_train, sample_weight=sample_weights)

pred_bagging = bagging_model.predict(x_test)

cm_bagging = confusion_matrix(y_test, pred_bagging, labels=[1, 0])
print("Bagging Confusion Matrix:\n", cm_bagging)

precision_bagging = precision_score(y_test, pred_bagging, pos_label=1)
recall_bagging = recall_score(y_test, pred_bagging, pos_label=1)

print(f"Precision (Class 1): {precision_bagging:.4f}")
print(f"Recall (Class 1): {recall_bagging:.4f}")

Bagging Confusion Matrix:
 [[  9   5]
 [ 59 144]]
Precision (Class 1): 0.1324
Recall (Class 1): 0.6429


In [ ]:
#Question 9:

#The precision counts from the class 1 matrix is 13.24% while the recall counts is 64.29%.
# I would say that it is a better model because with the sever imbalance that we saw with over 93% 0s and
# % 1s, a single model would overreact to sample weights and generate excessive false alarms.
# The bagging strategy would average out errors and shrink varaince. we can preserve high recall and reduce false positives.